# 02 · 智能路由结构化输出

本 notebook 演示如何使用 `StructuredRouter` 自动选择最佳结构化输出路线：

- 若模型支持原生 tool calling（OpenAI、DeepSeek、Qwen）→ 走 `with_structured_output` 路线
- 若不支持（如 MiniMax-M3）→ 自动降级到 `PydanticOutputParser + 输出清洗` 路线

In [1]:
# Cell 1: 导入依赖
import sys
sys.path.insert(0, '.')

from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from structured_router import StructuredRouter, invoke_structured, detect_tool_calling_support

In [2]:
# Cell 2: 定义模型与 Pydantic schema
model = ChatOpenAI(
    model="MiniMax-M3",
    api_key="sk-cp-GsTRU_GlFRX_WPY8cf8IqBTDtrVJP4zl4EWLZ8D7627lrDtV-MbWYnWWK7rmvT7ZRluuMtUL7VP2sCD2uq6SzkpIlY0EAJtCIHW2uSGAT4QKYxX9mJQmK8I",
    base_url="https://api.minimaxi.com/v1",
    temperature=0,
)

class Person(BaseModel):
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

print("模型与 schema 定义完成 ✅")

模型与 schema 定义完成 ✅


In [3]:
# Cell 3: 探针测试
ok = detect_tool_calling_support(model, verbose=True)
print(f"\n==> MiniMax-M3 是否支持 tool calling: {ok}")

[router] 探针：tool_calls=0, has_valid_args=False

==> MiniMax-M3 是否支持 tool calling: False


In [4]:
# Cell 4: 构建智能路由 chain
router = StructuredRouter(model, verbose=True)
chain = router.build(Person)
print(f"\n==> 选定路线: {router.route}")

[router] 探针：tool_calls=0, has_valid_args=False
[router] 选定路线: prompt_parser

==> 选定路线: prompt_parser


In [5]:
# Cell 5: 调用 chain
result = chain.invoke("张三是一名30岁的软件工程师")

print("=== 结构化结果 ===")
print(type(result).__name__, ":", result)
print()
print("姓名:", result.name)
print("年龄:", result.age)
print("职业:", result.occupation)

=== 结构化结果 ===
Person : name='张三' age=30 occupation='软件工程师'

姓名: 张三
年龄: 30
职业: 软件工程师


In [6]:
# Cell 6: 极简一行调用
person = invoke_structured(model, Person, "王五是一名28岁的产品经理")
print(person)

[router] 探针：tool_calls=0, has_valid_args=False
[router] 选定路线: prompt_parser


name='王五' age=28 occupation='产品经理'


In [7]:
# Cell 7: 调试 —— 强制指定路线
print("强制 prompt_parser 路线：")
chain_pp = StructuredRouter(model, verbose=True).build(Person, force="prompt_parser")
print(chain_pp.invoke("赵六是一名40岁的医生"))
print()

print("强制 function_calling 路线（预计会报错，因为 MiniMax-M3 不支持）：")
try:
    chain_fc = StructuredRouter(model, verbose=True).build(Person, force="function_calling")
    print(chain_fc.invoke("赵六是一名40岁的医生"))
except Exception as e:
    print(f"预期中的错误: {type(e).__name__}: {e}")

强制 prompt_parser 路线：
[router] 选定路线: prompt_parser


name='赵六' age=40 occupation='医生'

强制 function_calling 路线（预计会报错，因为 MiniMax-M3 不支持）：
[router] 选定路线: function_calling


预期中的错误: ValidationError: 1 validation error for Person
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='<think>The user has prov...进一步展开！😊', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid


## 总结

- `StructuredRouter` 会自动探测模型能力，选择最佳路线
- 以后换模型（不管是 GPT-4o 还是 MiniMax-M3）都不用改业务代码
- 需要代码复用，只要 `from structured_router import StructuredRouter, invoke_structured`